# Lab: Building ReAct Agents with LangChain and LangGraph
#
# Objectives:
# 1) Build a simple tool-calling agent with LangChain
# 2) Understand the ReAct loop: think -> act -> observe -> answer
# 3) Build a simple graph-based agent with LangGraph
# 4) Compare LangChain vs LangGraph

# Part 1 — LangChain Agent

In this part, we build a simple agent using LangChain.

The agent combines:
- a language model
- tools
- instructions

The model can decide whether it should:
- answer directly
- or call a tool first

This is a simple form of ReAct-style behavior:
1. Reason about the task
2. Act by calling a tool
3. Observe the tool result
4. Produce the final answer

# Importing Necessary Libraries

Here we import the necessary libraries for both LangChain and LangGraph:

- **TypedDict** and **Annotated** are used to define the structure of the state for LangGraph.
- **LangChain's core components** help us create an agent and define tools it can call.
- **ChatGroq** wraps the Groq model for LangChain integration.
- **LangGraph components** such as **StateGraph** and **ToolNode** are used for creating graph-based workflows that give us more control over agent behavior.

pip install langchain langgraph langchain-ollama langchain-groq numpy pandas matplotlib

In [ ]:
from typing import TypedDict, Annotated

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

ModuleNotFoundError: No module named 'langchain_groq'

# LangChain Model Setup

In this section, we initialize the **model** using **Ollama**. We're using the **qwen3:4b** model here, but you can replace it with any tool-compatible model you prefer.
This model will be used by the agent to generate responses based on user input.

We also set **temperature=0**, meaning the output will be deterministic (no randomness).

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434",
    temperature=0
)

# Tool Definitions

Here, we define two tools for the agent:

1. **Calculator**: This tool takes a string representing a mathematical expression and evaluates it (e.g., '15 * 8').
2. **Course Lookup**: This tool returns short course information based on a topic (e.g., "LangChain" or "LangGraph").

The **@tool** decorator in LangChain tells the agent that these are callable functions the agent can invoke.

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a simple math expression like '25 * 4 + 10'."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def course_lookup(topic: str) -> str:
    """Return short course information for a requested topic."""
    data = {
        "langchain": "LangChain is a framework for building LLM applications with prompts, tools, chains, and agents.",
        "langgraph": "LangGraph is a framework for building stateful, graph-based agent workflows.",
        "react": "ReAct stands for Reason plus Act. The model reasons, uses tools, observes results, and then continues."
    }
    return data.get(topic.lower(), f"No course information found for '{topic}'.")

In [ ]:
print(calculator.invoke("5 * 12 + 3"))
print(course_lookup.invoke("langgraph"))

Result: 63
LangGraph is a framework for building stateful, graph-based agent workflows.


In [ ]:
agent = create_agent(
    model=model,
    tools=[calculator, course_lookup],
    system_prompt=(
        "You are a helpful study assistant. "
        "Use tools when needed. "
        "If the user asks about math, use the calculator tool. "
        "If the user asks about course topics, use the course_lookup tool."
    ),
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is 15 * 8, and what is LangGraph?"}
        ]
    }
)

response

{'messages': [HumanMessage(content='What is 15 * 8, and what is LangGraph?', additional_kwargs={}, response_metadata={}, id='2a522ec1-5f84-4ff2-9193-4e4f9b4651e8'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-14T16:21:26.846185Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24951655000, 'load_duration': 3043310291, 'prompt_eval_count': 234, 'prompt_eval_duration': 3003903542, 'eval_count': 554, 'eval_duration': 18607265427, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019d8ccb-d532-7313-b278-bce4b73bdd9c-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '15 * 8'}, 'id': 'd44cff8c-7760-451d-84b5-dbbab9b665fe', 'type': 'tool_call'}, {'name': 'course_lookup', 'args': {'topic': 'LangGraph'}, 'id': 'bd3b99fe-bab9-4233-9ed2-134afd71ff70', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 554, 'total_tokens': 788}),
 

In [ ]:
print(response["messages"][-1].content)  # Display the final answer

The result of $ 15 \times 8 $ is **120**.  

LangGraph is a framework for building **stateful, graph-based agent workflows**.


# LangGraph Setup

In this section, we define the **LangGraph** workflow.

1. **State**: The state represents the conversation history, which is stored as messages.
2. **Nodes**:
   - The **chatbot node** calls the model to get a response.
   - The **tool node** runs the tools (calculator and course lookup).
3. **Graph Flow**: The flow starts at the chatbot node. If the model needs tools, it calls the tool node, executes the tools, and then loops back to the chatbot node.

This is a **graph-based approach**, which gives us more control over the decision-making process of the agent.

# Importing Required Libraries for LangGraph

In this section, we import the necessary libraries for LangGraph to create a **graph-based agent**:

1. **`TypedDict` and `Annotated`**:
   - These are used to define the **structure** of the state in the **LangGraph**.
   - `TypedDict` allows us to specify that the state will be a dictionary with predefined keys, and `Annotated` is used to annotate that `messages` will hold the conversation history.

2. **`StateGraph`, `START`, `END`**:
   - **`StateGraph`** is the core class that allows us to define a graph of nodes (each node represents an action, such as calling the model or using a tool).
   - **`START`** and **`END`** represent the entry and exit points in the graph. The flow of the agent starts at `START` and ends at `END`.

3. **`add_messages`**:
   - This is a utility function to handle and manage **messages** (the conversation history). It allows us to add new messages to the state during the agent's workflow.

4. **`ToolNode` and `tools_condition`**:
   - **`ToolNode`** represents a node in the graph that executes **tools** (like our calculator or course lookup). It ensures that the tools are called when necessary.
   - **`tools_condition`** is a condition used to determine when to move from one node to another. In this case, it helps us decide when to execute a tool, based on the conversation flow.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
tools = [calculator, course_lookup]
model_with_tools = model.bind_tools(tools)

In [ ]:
def chatbot_node(state: AgentState):
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile()

In [ ]:
result = graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is 15 * 8, and what is LangGraph?"}
        ]
    }
)

In [ ]:
for msg in result["messages"]:
    print(msg)
    print("-" * 80)

content='What is 15 * 8, and what is LangGraph?' additional_kwargs={} response_metadata={} id='7763a848-06c9-4ec2-84cb-b82426e9bd8b'
--------------------------------------------------------------------------------
content='' additional_kwargs={} response_metadata={'model': 'qwen3:4b', 'created_at': '2026-04-14T13:12:26.590861Z', 'done': True, 'done_reason': 'stop', 'total_duration': 27192680916, 'load_duration': 139840333, 'prompt_eval_count': 196, 'prompt_eval_duration': 5828114500, 'eval_count': 575, 'eval_duration': 20788427048, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'} id='lc_run--019d8c1e-c28a-7422-95ee-3892061a0aca-0' tool_calls=[{'name': 'calculator', 'args': {'expression': '15 * 8'}, 'id': '55f40bce-e78e-4471-bb3f-2aa0941b5bbf', 'type': 'tool_call'}, {'name': 'course_lookup', 'args': {'topic': 'LangGraph'}, 'id': '0b6c2a22-a4ed-4cbf-94c5-c01890638c05', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 196, 'output_tokens':

# Reflection Questions
#
# 1) What is the difference between a normal LLM call and an agent?
# 2) Why is ReAct useful?
# 3) What is easier to use: LangChain or LangGraph?
# 4) When would you prefer LangGraph over LangChain?
# 5) What are the limitations of this simple agent?

ANSWERS

1. An LLM call is just a prompt and response loop once, without saving any history, where each call is independent of another. An agent is a whole LLM which saves all LLM calls and stores chat history, can make decisions, operate in multiple steps, etc...

2. Because it allows the model to reason and look back at decisions in a loop, allowing it to use tools to verify it's choices and improve it's decision making.

3. LangChain is easier as it steps are super simple to follow and it's workflow is linear. LangGraph should be used when completing complex tasks, where the workflow involves loops, conditional logic and agent like behavior.

4. An example of this is a SW engineering project where the workflows are super long and complex, involving many agents and lots of conditional loops.

5. An agent is limited by the knowledge of the underlying model it uses. As the world progresses many things change and need adaptation. It is very expensive to keep retraining the model to fit the new world's standard.

# Task: Create Your Own Graph with a Custom Chatbot Tool
# Objective:


# Your task is to create a graph-based workflow using LangGraph, with your own custom chatbot tool.

# Notes:
#
# - LangChain is the easier, higher-level framework for building agents.
# - LangGraph is the lower-level orchestration framework for building
#   stateful workflows with nodes, edges, and shared state.
# - ReAct means the model can reason, use a tool, observe the result,
#   and continue until it can answer.